In [ ]:
# Cell 1: Install dependencies
!pip install -q scanpy anndata igraph leidenalg scikit-learn scipy requests
!pip install -q cellxgene-census

In [ ]:
# Cell 2: Mount Drive and upload project files
from google.colab import drive, files
drive.mount('/content/drive')

PRETRAIN_DIR = '/content/drive/MyDrive/CellJEPA_results/kidney_pretrain/'
PERT_DIR     = '/content/drive/MyDrive/CellJEPA_results/perturbation/'
import os
os.makedirs(PRETRAIN_DIR, exist_ok=True)
os.makedirs(PERT_DIR, exist_ok=True)

# Upload project files:
# cell_jepa.py, compare_perturbation.py, losses.py,
# perturb_metrics.py, preprocessing.py, trainer.py, pretrain_kidney.py
# files.upload()


In [ ]:
# Cell 3: Kidney pre-training (~8-10 hours on A100, 4 epochs, ~200k cells)
import subprocess, time, os, glob

REPO_DIR     = '/content'
PRETRAIN_DIR = '/content/drive/MyDrive/CellJEPA_results/kidney_pretrain/'

existing = sorted(glob.glob(os.path.join(PRETRAIN_DIR, 'kidney_pretrain_epoch*.pt')))
resume_arg = []
if existing:
    latest = existing[-1]
    print(f'Found checkpoint: {latest} — will resume from here')
    resume_arg = ['--resume', latest]
else:
    print('No checkpoint found — starting pre-training from scratch')

t0 = time.time()
cmd = [
    'python3', '-u', os.path.join(REPO_DIR, 'pretrain_kidney.py'),
    '--device', 'cuda',
    '--n_epochs', '4',
    '--batch_size', '32',   # 128 causes CUDA OOM; 32 matches paper and fits A100
    '--drive_dir', PRETRAIN_DIR,
] + resume_arg

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True, bufsize=1
)

import threading
def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join();  t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} — see stderr above ***')
else:
    print(f'\nPre-training done in {(time.time()-t0)/60:.1f} min')
    print(f'Final checkpoint: {PRETRAIN_DIR}kidney_pretrain_final.pt')

In [ ]:
# Cell 4: Smoke test — runs in ~1 min, prints dataset structure
# Run this first to confirm the dataset loads correctly before the full run.
import subprocess, os

REPO_DIR = '/content/VirtualCellJEPA'
result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'compare_perturbation.py'),
     '--smoke_test', '--device', 'cpu'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 5: Full 2x2 ablation with kidney pre-trained backbone (~30-60 min on A100)
# Requires kidney pre-training (Cell 3) to have completed first.
import subprocess, time, os

REPO_DIR     = '/content/VirtualCellJEPA'
PRETRAIN_DIR = '/content/drive/MyDrive/CellJEPA_results/kidney_pretrain/'
checkpoint   = os.path.join(PRETRAIN_DIR, 'kidney_pretrain_final.pt')

if not os.path.exists(checkpoint):
    print(f'WARNING: checkpoint not found at {checkpoint}')
    print('Run Cell 3 (kidney pre-training) first, or set checkpoint path manually.')
else:
    t0 = time.time()
    proc = subprocess.Popen(
        ['python3', '-u', os.path.join(REPO_DIR, 'compare_perturbation.py'),
         '--device', 'cuda',
         '--n_epochs', '15',
         '--batch_size', '64',
         '--pretrain_checkpoint', checkpoint],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 6: Display results and plot
import os, re
import numpy as np
import matplotlib.pyplot as plt

if os.path.exists('results_perturbation.txt'):
    print(open('results_perturbation.txt').read())
else:
    print('No results file found — did Cell 5 complete?')

def parse_pert_results(path):
    results = {}
    if not os.path.exists(path): return results
    for line in open(path):
        nums = re.findall(r'-?\d+\.\d+', line)
        if len(nums) >= 4 and not line.strip().startswith(('=', '-', 'C', 'R')):
            label = line[:36].strip()
            if label:
                results[label] = [float(x) for x in nums[:4]]
    return results

results = parse_pert_results('results_perturbation.txt')
if results:
    labels = list(results.keys())
    data = np.array(list(results.values()))
    metrics = ['Pearson', 'Pearson delta', 'Top-20 DEG delta', 'MSE']
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
    x = np.arange(len(labels))
    bw = 0.18
    offsets = np.linspace(-1.5, 1.5, 4) * bw

    fig, ax = plt.subplots(figsize=(13, 5))
    for i, (m, c, off) in enumerate(zip(metrics, colors, offsets)):
        bars = ax.bar(x + off, data[:, i], bw, label=m, color=c, alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bw/2, h + 0.003, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=7, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('Score')
    ax.set_title('CellJEPA Perturbation Prediction — Adamson 2016\n2x2 Ablation: Objective x JEPA (kidney pretrained)')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig('perturbation_ablation.png', dpi=150)
    plt.show()
    print('Saved perturbation_ablation.png')


In [ ]:
# Cell 7: Save to Drive
import shutil, os

PERT_DIR = '/content/drive/MyDrive/CellJEPA_results/perturbation/'
for f in ['results_perturbation.txt', 'perturbation_ablation.png', 'adamson_genes.json']:
    if os.path.exists(f):
        shutil.copy(f, PERT_DIR)
        print(f'Copied {f}')
print(f'Done. Files in {PERT_DIR}')
